In [3]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
import pandas as pd
from fonts_config import set_computer_modern, truncate_colormap
set_computer_modern()
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False
from matplotlib import cm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
import glob
import os


In [4]:

# ------------------------------------------------------------
# paths
# ------------------------------------------------------------
n_best=2572
ele = xr.open_dataset("../../FesmData/Vinther2009_elevations/vinther2009.nc")
S = xr.open_dataset("../scoring/scores/scores_final.nc")
ds_ensemble = xr.open_dataset("../output/ensemble_elevations.nc")
valid_sim_indices = ds_ensemble.sim

best = ds_ensemble.sel(sim=n_best)
time = best["time"].values * 1e-3  # kyr BP
z_ngri = best.z_srf.sel(ice_core="ngrip").values
z_grip = best.z_srf.sel(ice_core="grip").values
z_camp = best.z_srf.sel(ice_core="camp_century").values
z_dye3 = best.z_srf.sel(ice_core="dye3").values

In [5]:
cores = {
    "ngrip":        {"label": "NGRIP",        "y_text": 2650, "data": z_ngri, "letter": "(a) NGRIP"},
    "grip":         {"label": "GRIP",         "y_text": 3050, "data": z_grip, "letter": "(b) GRIP"},
    "camp_century": {"label": "Camp Century", "y_text": 1850, "data": z_camp, "letter": "(c) Camp Century"},
    "dye3":         {"label": "DYE3",         "y_text": 1800, "data": z_dye3, "letter": "(d) DYE-3"}
}

s_values = S.S.values 
colors = ["#D9DAB3", "#3C9D9F", "#002665"]
nodes = [0, 0.3, 1.0]
cmap = mcolors.LinearSegmentedColormap.from_list("custom_warm", list(zip(nodes, colors)))
sorted_sim_indices = S.sim.sortby(S.S).values
norm = mcolors.Normalize(vmin=s_values.min()*100, vmax=s_values.max()*100)


fig, axes = plt.subplots(2, 2, figsize=(7, 7), sharex=True, constrained_layout=False)
axs = axes.flatten()
axes[0, 0].sharey(axes[0, 1])
axes[1, 0].sharey(axes[1, 1])
for i, (icec, info) in enumerate(cores.items()):
    ax = axs[i]
    
    for j, sim_idx in enumerate(sorted_sim_indices):
        color_s = cmap(norm(S.S.sel(sim=sim_idx).values*100))
        ax.plot(ds_ensemble.time * 1e-3, ds_ensemble.z_srf.sel(sim=sim_idx, ice_core=icec)*1e-3, 
                "-", color=color_s, alpha=0.1, zorder=0)

    ax.plot(time, info["data"]*1e-3, color="black", label=f"Best simulation" if i == 3 else None)

    vint = ele.sel(ice_core=icec)
    ax.fill_between(vint.time*1e-3, vint.z_srf*1e-3 - vint.error*1e-3, vint.z_srf*1e-3 + vint.error*1e-3, 
                    color="blue", alpha=0.2, label="Vinther et al. (2009)" if i == 3 else None)
    ax.plot(vint.time*1e-3, vint.z_srf*1e-3, color="blue")

    ax.text(-11.5, info["y_text"], info["label"], ha="left", va="top", fontweight="bold")
    ax.text(0.95, 0.95, info["letter"], transform=ax.transAxes, va='top', ha='right')
    
    ax.grid(alpha=0.2, zorder=1)
    ax.set_xlim(-12, 0.1) 
    if i % 2 == 0: 
        ax.set_ylabel("Elevation (km)")
    else:
        plt.setp(ax.get_yticklabels(), visible=False)
    if i >= 2: 
        ax.set_xlabel("Time (kyr BP)")
        ax.set_ylim(1.75, 3)
    else:
        ax.set_ylim(2.6, 3.5)


handles, labels = axs[3].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
plt.subplots_adjust(top=0.93, bottom=0.2, left=0.12, right=0.95, hspace=0.07, wspace=0.05)

cbar_ax = fig.add_axes([0.25, 0.08, 0.5, 0.03])
fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax)
cb = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, orientation='horizontal')
cb.set_label('Global score S (x10²)')


plt.savefig(f"../figs_final/fig4_{n_best}_elevations_ensemble.pdf", dpi=300)
plt.close()